# Section Anything: quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omniwot/section-anything/blob/main/notebooks/quickstart.ipynb)

Find the real sections of a PDF: every heading, its level, the pages it spans and the exact text under it. This notebook installs the library from PyPI, downloads one public paper and runs **Free** mode (no model, no key, nothing leaves the machine). An optional last step runs **Quick** mode with your own OpenAI key.

Paper used: *Visual Instruction Tuning*, Liu et al., [arXiv:2304.08485](https://arxiv.org/abs/2304.08485) (v2), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). It is downloaded from arXiv when you run the notebook; the outputs below show its headings and short excerpts, unmodified.

In [ ]:
%pip install -q section-anything

## 1. Download the paper

In [ ]:
import urllib.request

URL = "https://arxiv.org/pdf/2304.08485v2"  # Visual Instruction Tuning, CC BY 4.0
req = urllib.request.Request(URL, headers={"User-Agent": "section-anything-quickstart"})
with urllib.request.urlopen(req, timeout=60) as r, open("paper.pdf", "wb") as f:
    f.write(r.read())
print("saved paper.pdf")

saved paper.pdf


## 2. Free mode: the heading tree

Free mode uses the PDF's bookmarks or printed table of contents, then typography and numbering rules. The document title is `#`; every other heading is `#` × (level + 1).

In [ ]:
import section_anything

tree = section_anything.section("paper.pdf", mode="free")
print(tree.to_markdown())

# Visual Instruction Tuning
### Abstract
### Introduction
## 2 Related Work
## 3 GPT-assisted Visual Instruction Data Generation
## 4 Visual Instruction Tuning
### 4.1 Architecture
### 4.2 Training
## 5 Experiments
### 5.1 Multimodal Chatbot
### 5.2 ScienceQA
## 6 Conclusion
### References
### A Broader Impact
### B More Results
### C Training Details
### D Assets
### E Data
### F Prompts



In [ ]:
tree.summary()

{'mode': 'free',
 'model': None,
 'pages': 25,
 'sections': 19,
 'flagged': 1,
 'calls': 0,
 'input_tokens': 0,
 'output_tokens': 0,
 'cost_usd': 0.0,
 'seconds': 3.73,
 'downgraded': []}

## 3. One section: level, pages and character span

`tree.sections` is a list of dicts in reading order. `start` and `end` are character offsets into `tree.text`, so the section's own text is one slice.

In [ ]:
sec = next(s for s in tree.sections if s["number"] == "4.1")
print(sec)
print()
print(tree.text[sec["start"]:sec["end"]][:500], "...")

{'number': '4.1', 'title': 'Architecture', 'level': 2, 'page_start': 4, 'page_end': 4, 'start': 13633, 'end': 15017, 'flags': []}

4.1 Architecture
The primary goal is to effectively leverage the capabilities of both the pre-trained LLM and visual
model. The network archtecture is illustrated in Figure 1. We choose Vicuna [9] as our LLM fϕ(·)
parameterized by ϕ, as it has the best instruction following capabilities in language tasks among
publicly available checkpoints [48, 9, 38].
Figure 1: LLaVA network architecture.
For an input image Xv, we consider the pre-trained CLIP visual encoder ViT-L/14 [40], which
provides the v ...


The same section as Markdown with its text reflowed into paragraphs and running headers and footers removed. `to_markdown(body=True)` is what `section-anything paper.pdf --body` prints.

In [ ]:
body_md = tree.to_markdown(body=True)
i = body_md.index(f"{sec['number']} {sec['title']}")
start = body_md.rfind("\n", 0, i) + 1   # start of the heading line
end = body_md.find("\n#", i)            # next heading
print(body_md[start:end][:900], "...")

### 4.1 Architecture

The primary goal is to effectively leverage the capabilities of both the pre-trained LLM and visual model. The network archtecture is illustrated in Figure 1. We choose Vicuna [9] as our LLM fϕ(·) parameterized by ϕ, as it has the best instruction following capabilities in language tasks among publicly available checkpoints [48, 9, 38].

Figure 1: LLaVA network architecture.

For an input image Xv, we consider the pre-trained CLIP visual encoder ViT-L/14 [40], which provides the visual feature Zv = g(Xv). The grid features before and after the last Transformer layer are considered in our experiments. We consider a simple linear layer to connect image features into the word embedding space. Specifically, we apply a trainable projection matrix W to convert Zv into language embedding tokens Hv, which have the same dimensionality as the word embedding space in the langu ...


The command line gives the same results; `--format json` adds the spans, mode and cost:

In [ ]:
!section-anything paper.pdf --format json -q | head -n 28

{
  "section_anything": "0.1.0",
  "mode": "free",
  "model": null,
  "pages": 25,
  "sections": [
    {
      "number": "",
      "title": "Visual Instruction Tuning",
      "level": 1,
      "page_start": 1,
      "page_end": 1,
      "start": 68,
      "end": 252,
      "flags": [
        "doc_title"
      ]
    },
    {
      "number": "",
      "title": "Abstract",
      "level": 2,
      "page_start": 1,
      "page_end": 1,
      "start": 253,
      "end": 1503,
      "flags": []
    },


## 4. Optional: Quick mode with your own OpenAI key

Quick mode sends only the candidate heading lines (plus a short peek at the next line) to a small model, which keeps the real headings and sets their levels; about $0.07 per 1,000 pages with the default model. The key is read with `getpass`, passed to this one call and not stored or written to the environment. `max_usd` caps this document's cost; over the cap the run falls back to Free.

Leave the prompt empty to skip.

In [ ]:
from getpass import getpass

key = getpass("OpenAI API key (empty to skip): ")
if key:
    quick = section_anything.section("paper.pdf", mode="quick", api_key=key, max_usd=0.05)
    print(quick.to_markdown())
    print(quick.summary())
else:
    print("Skipped.")
del key

## Next steps

- Other OpenAI-compatible endpoints (Azure OpenAI, OpenRouter, vLLM, a LiteLLM proxy, ...) and what they must support: [Providers](https://github.com/Omniwot/section-anything/blob/main/packages/section-anything/README.md#providers).
- Fully local: install [Ollama](https://ollama.com) and a small model, then `section-anything paper.pdf --mode local:qwen3:4b` (nothing leaves the machine).
- `mode="standard"`: a model reads every line and marks where sections start and end (higher accuracy, higher cost, still capped per document).
- Benchmarks, settings and limitations: [README](https://github.com/Omniwot/section-anything).